# FNO Model Evaluation — GCS-enabled

All source settings live in **`eval_gcs_config.yaml`** (or the `EVAL_GCS_CONFIG`
environment variable).  Edit that file to change model/data sources, backend, and
checkpoint type — no code changes needed.

**Eval modes:**
| Mode | Description |
|---|---|
| `basic`  | Truth / prediction image grids per η bin at selected τ steps |
| `cont`   | CDF-based contour overlays (3 τ snapshots per η bin) |
| `metric` | Full metric suite: nRMSE · fRMSE · MELR · WLR dashboard |

Select a mode, adjust parameters, then click **Run Evaluation**.

In [ ]:
import io
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg')          # non-interactive backend: no auto-display on figure
import matplotlib.pyplot as plt  # creation, plt.close(), or plt.show()
plt.ioff()                     # belt-and-suspenders: keep interactive mode off
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output, Image

sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
import eval_gcs_setup as _setup
try:
    _ctx = _setup.run_all()
except Exception as _e:
    raise RuntimeError(f'Setup failed: {_e}') from _e

eval_cfg       = _ctx.eval_cfg
train_cfg      = _ctx.train_cfg
backend        = _ctx.backend
data_test      = _ctx.data_test
model_out_phys = _ctx.model_out_phys
y_true_phys    = _ctx.y_true_phys
eta_coords     = _ctx.eta_coords
tau_out_coords = _ctx.tau_out_coords
x_coords       = _ctx.x_coords
y_coords       = _ctx.y_coords
xy_extent      = _ctx.xy_extent
dtau           = _ctx.dtau
tau_min        = _ctx.tau_min
neta_eff       = _ctx.neta_eff
ntau_out       = _ctx.ntau_out

In [ ]:
_norm     = train_cfg.get('normalization', {})
norm_name = _norm.get('name', 'none') if isinstance(_norm, dict) else str(_norm)
_n_test        = data_test.shape[0]
_default_mode  = eval_cfg.get('eval_mode', 'basic')
_default_thresh = float(eval_cfg.get('active_threshold', 0.3))

_ws = {'description_width': 'initial'}
_w300 = widgets.Layout(width='300px')
_w350 = widgets.Layout(width='350px')
_w400 = widgets.Layout(width='400px')
_w500 = widgets.Layout(width='500px')

eval_mode_ui = widgets.Dropdown(
    options=['basic', 'cont', 'metric'],
    value=_default_mode,
    description='Eval mode:',
    style=_ws,
)
active_thresh_ui = widgets.BoundedFloatText(
    value=_default_thresh, min=0.0, max=5.0, step=0.05,
    description='Active ε threshold (GeV/fm³, 0=off):',
    layout=_w300, style=_ws,
)

# basic / metric controls
event_ui = widgets.BoundedIntText(
    value=0, min=0, max=_n_test - 1, step=1,
    description='Event index for plots:', style=_ws,
)
channel_num_ui = widgets.BoundedIntText(
    value=0, min=0, max=3, step=1,
    description='Channel (0=ε, 1=vy, 2=vx, 3=vη):', style=_ws,
)
channel_drop_ui = widgets.Dropdown(
    options=['0 (ε)', '1 (vx)', '2 (vy)', '3 (vη)'],
    value='0 (ε)', description='Channel:', style=_ws,
)
tau_idx_ui = widgets.Text(
    value='5,10,15,-1',
    description='τ output indices (comma-sep, -1=last):',
    layout=_w350, style=_ws,
)
ieta_ui = widgets.BoundedIntText(
    value=0, min=0, max=neta_eff - 1, step=1,
    description='η bin index:', style=_ws,
)
spec_tau_ui = widgets.Text(
    value='0,-1',
    description='τ indices for energy spectrum (comma-sep):',
    layout=_w350, style=_ws,
)
save_json_ui = widgets.Checkbox(
    value=False, description='Save summary to JSON', style=_ws,
)

# cont controls
cont_event_ui = widgets.IntSlider(
    min=0, max=_n_test - 1, value=0, step=1,
    description='Event:', layout=_w400, style=_ws,
)
cont_ieta_ui = widgets.IntSlider(
    min=0, max=neta_eff - 1, value=0, step=1,
    description='η bin:', layout=_w400, style=_ws,
)
cont_channel_ui = widgets.Dropdown(
    options=['0 — ε (energy)', '1 — vx', '2 — vy', '3 — vη'],
    value='0 — ε (energy)', description='Feature / channel:', style=_ws,
)
# descriptions moved to Label widgets in the layout so they are never clipped
contour_iT_ui = widgets.Text(
    value='10,-1',
    description='',
    layout=widgets.Layout(width='120px'),
)
cdfrats_ui = widgets.Text(
    value='0.05,0.1,0.2,0.3,0.4,0.6',
    description='',
    layout=widgets.Layout(width='260px'),
)
rescale_energy_ui = widgets.Checkbox(
    value=(norm_name != 'none'),
    description='Rescale energy: (e/τ)^(4/3)', style=_ws,
)

In [ ]:
# --- mode-specific control panels ------------------------------------
_basic_panel = widgets.HBox([
    event_ui, ieta_ui, tau_idx_ui, channel_num_ui,
])
_cont_panel = widgets.VBox([
    widgets.HBox([cont_event_ui, cont_ieta_ui, cont_channel_ui]),
    widgets.HBox([
        widgets.VBox([
            widgets.Label('2nd and 3rd τ-step indices (comma-sep; 1st panel always 0; -1=last non-zero):'),
            contour_iT_ui,
        ]),
        widgets.VBox([
            widgets.Label('CDF ratios for contour levels (comma-sep):'),
            cdfrats_ui,
        ]),
    ]),
    widgets.HBox([
        widgets.HTML(f'<b>Normalizer:</b> <code>{norm_name}</code>&nbsp;&nbsp;'),
        rescale_energy_ui,
    ]),
])
_metric_panel = widgets.VBox([
    widgets.HBox([channel_drop_ui, ieta_ui]),
    widgets.HBox([spec_tau_ui, save_json_ui]),
])

_top_row       = widgets.HBox([eval_mode_ui, active_thresh_ui])
_controls_body = widgets.VBox([_basic_panel])


def _update_controls(change):
    mode = change['new']
    if mode == 'basic':
        _controls_body.children = [_basic_panel]
    elif mode == 'cont':
        _controls_body.children = [_cont_panel]
    else:
        _controls_body.children = [_metric_panel]


eval_mode_ui.observe(_update_controls, names='value')
_update_controls({'new': eval_mode_ui.value})

# --- run button + output area ----------------------------------------
_run_btn  = widgets.Button(
    description='Run Evaluation',
    button_style='primary',
    icon='play',
    layout=widgets.Layout(width='200px', margin='8px 0'),
)
_plot_out = widgets.Output()


def _show(fig):
    """Render fig to a PNG buffer and display as Image, then close the figure.

    With matplotlib's Agg backend active (set in cell-1):
      - plt.subplots() never auto-displays
      - plt.close()   is a pure cleanup with no display side-effects
      - plt.show()    is a no-op
    So display(Image(...)) here is the one and only render path."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=fig.dpi, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.read()))


def _run_eval(b):
    mode = eval_mode_ui.value
    with _plot_out:
        clear_output(wait=True)

        # ---- basic plots --------------------------------------------
        if mode == 'basic':
            _event   = int(event_ui.value)
            _ieta    = int(ieta_ui.value)
            _channel = int(channel_num_ui.value)
            _raw_idx     = [t.strip() for t in tau_idx_ui.value.split(',') if t.strip()]
            _tau_indices = [int(t) % ntau_out for t in _raw_idx]
            _tau_indices = [t for t in _tau_indices if 0 <= t < ntau_out]
            _ch_label    = ['e (GeV/fm³)', 'vx', 'vy', 'vη'][_channel] if _channel < 4 else f'ch{_channel}'
            _eta_phys    = eta_coords[_ieta]

            _fig, _axes = plt.subplots(
                2, len(_tau_indices),
                figsize=(4.5 * len(_tau_indices), 7), squeeze=False,
            )
            _fig.suptitle(f'η = {_eta_phys:.2f}  (event {_event})', fontsize=14, y=1.01)
            for _col, _t in enumerate(_tau_indices):
                _tau_phys = tau_out_coords[_t]
                _true = y_true_phys[_event, _channel, :, :, _ieta, _t]
                _pred = model_out_phys[_event, _channel, :, :, _ieta, _t]
                _vmin, _vmax = min(_true.min(), _pred.min()), max(_true.max(), _pred.max())
                _kw = dict(origin='lower', extent=xy_extent, aspect='equal',
                           vmin=_vmin, vmax=_vmax, cmap='hot')
                for _row, (_dat, _lbl) in enumerate([
                    (y_true_phys, 'Truth'), (model_out_phys, 'FNO'),
                ]):
                    _ax = _axes[_row, _col]
                    _im = _ax.imshow(_dat[_event, _channel, :, :, _ieta, _t].T, **_kw)
                    _ax.set_title(f'{_lbl}   τ = {_tau_phys:.3f} fm/c', fontsize=10)
                    _ax.set_xlabel('x (fm)')
                    _ax.set_ylabel('y (fm)')
                    plt.colorbar(_im, ax=_ax, label=_ch_label, fraction=0.046, pad=0.04)
            _fig.tight_layout()
            _show(_fig)

        # ---- cont plots ---------------------------------------------
        elif mode == 'cont':
            from skinnycontour import plot_three_bins_contour
            _event          = int(cont_event_ui.value)
            _channel        = int(cont_channel_ui.value.split()[0])
            _rescale_energy = bool(rescale_energy_ui.value)
            _raw_iT = [t.strip() for t in contour_iT_ui.value.split(',') if t.strip()]
            if len(_raw_iT) < 2:
                _raw_iT = (_raw_iT + ['-1', '-1'])[:2]
            elif len(_raw_iT) > 2:
                _raw_iT = _raw_iT[:2]
            _contour_iT = (0,) + tuple(int(t) for t in _raw_iT)
            _cdfrats    = [float(x.strip()) for x in cdfrats_ui.value.split(',') if x.strip()]
            _tau0       = tau_min + dtau
            _ieta       = int(cont_ieta_ui.value)
            _eta_phys   = eta_coords[_ieta]

            _fig, _axes = plt.subplots(1, 3, figsize=(16.0, 4.5))
            plot_three_bins_contour(
                model_out_phys, y_true_phys,
                iparam=_channel, event=_event,
                tau0=_tau0, taudelta=dtau, iT=_contour_iT,
                cdfrats=_cdfrats, ieta=_ieta, tag=f'η={_eta_phys:.2f}',
                rescale_energy=_rescale_energy,
                x_coords=x_coords, y_coords=y_coords,
                fig=_fig, axes=_axes,
            )
            _show(_fig)

        # ---- metric plots -------------------------------------------
        else:
            import json
            from model_eval import ModelEval

            _thresh         = active_thresh_ui.value if active_thresh_ui.value > 0 else None
            _ch             = int(channel_drop_ui.value.split()[0])
            _tau_idx        = [int(t.strip()) for t in spec_tau_ui.value.split(',') if t.strip()]
            _sfx            = ' — MLX' if backend == 'mlx' else ''
            _metric_backend = eval_cfg.get('metric_backend', 'numpy')

            _ieta     = int(np.argmin(np.abs(eta_coords - 0.0)))
            _itau_3fm = int(np.argmin(np.abs(tau_out_coords - 3.0)))

            _evaluator = ModelEval(
                pred=model_out_phys, true=y_true_phys,
                tau_coords=tau_out_coords, eta_coords=eta_coords,
                channel_names=['ε', 'vx', 'vy', 'vη'],
                active_threshold=_thresh,
                backend=_metric_backend,
            )

            _fig_dash = _evaluator.plot_dashboard(
                channel=_ch, ieta=_ieta, tau_indices=_tau_idx, itau=_itau_3fm,
            )
            _show(_fig_dash)

            _nrmse = _evaluator.nrmse()
            _fig2, _axes2 = plt.subplots(1, _evaluator.C, figsize=(5 * _evaluator.C, 3.5))
            if _evaluator.C == 1:
                _axes2 = [_axes2]
            for _c, _ax in enumerate(_axes2):
                _im = _ax.pcolormesh(tau_out_coords, eta_coords, _nrmse[_c],
                                     cmap='hot_r', shading='auto')
                plt.colorbar(_im, ax=_ax, label='nRMSE')
                _ax.set_xlabel('τ (fm/c)')
                _ax.set_ylabel('η')
                _ax.set_title(f'nRMSE — {_evaluator.channel_names[_c]}')
            _fig2.suptitle(f'nRMSE heat map (η × τ){_sfx}', fontsize=12)
            _fig2.tight_layout()
            _show(_fig2)

            _frmse = _evaluator.frmse()
            _fig3, _axes3 = plt.subplots(1, 3, figsize=(15, 3.5))
            for _ax, (band, cmap) in zip(_axes3,
                    [('low', 'Blues'), ('mid', 'Oranges'), ('high', 'Reds')]):
                _im = _ax.pcolormesh(tau_out_coords, eta_coords, _frmse[band][_ch],
                                     cmap=cmap, shading='auto')
                plt.colorbar(_im, ax=_ax, label='fRMSE')
                _ax.set_xlabel('τ (fm/c)')
                _ax.set_ylabel('η')
                _ax.set_title(f'fRMSE ({band}) — {_evaluator.channel_names[_ch]}')
            _fig3.suptitle(f'Spectral RMSE heat maps (η × τ){_sfx}', fontsize=12)
            _fig3.tight_layout()
            _show(_fig3)

            _s = _evaluator.summary()
            display(widgets.HTML(
                '<h3>Scalar summary</h3>'
                '<table style="border-collapse:collapse">'
                '<tr><th style="text-align:left;padding:4px 12px">Metric</th>'
                '<th style="text-align:right;padding:4px 12px">Value</th></tr>'
                + ''.join(
                    f'<tr><td style="padding:3px 12px"><code>{k}</code></td>'
                    f'<td style="padding:3px 12px;text-align:right">{v:.6g}</td></tr>'
                    for k, v in _s.items()
                )
                + '</table>'
            ))

            if save_json_ui.value:
                _out_path = Path(eval_cfg.get('odir', './out')) / (
                    f'metrics_eval{"_mlx" if backend == "mlx" else ""}.json'
                )
                _out_path.parent.mkdir(parents=True, exist_ok=True)
                with open(_out_path, 'w') as _f:
                    json.dump(_s, _f, indent=2)
                display(widgets.HTML(
                    '<div style="background:#d4edda;border:1px solid #c3e6cb;'
                    'padding:8px;border-radius:4px;margin-top:8px">'
                    f'Saved → <code>{_out_path}</code></div>'
                ))

        # ---- relative L² error vs τ (basic + cont) ------------------
        if mode != 'metric':
            _ch    = int(channel_num_ui.value)
            _denom = np.mean(y_true_phys[:, _ch] ** 2, axis=(0, 1, 2))
            _num   = np.mean(
                (model_out_phys[:, _ch] - y_true_phys[:, _ch]) ** 2, axis=(0, 1, 2)
            )
            _rel_l2 = np.sqrt(_num / np.maximum(_denom, 1e-30))
            _sfx    = ' — MLX' if backend == 'mlx' else ''
            _fig4, _ax4 = plt.subplots(figsize=(8, 4))
            for _ie in range(neta_eff):
                _ax4.plot(tau_out_coords, _rel_l2[_ie], label=f'η={eta_coords[_ie]:.2f}')
            _ax4.set_xlabel('τ (fm/c)')
            _ax4.set_ylabel('Relative L² error')
            _ax4.set_title(f'Relative L² error vs τ (all η bins, test set){_sfx}')
            _ax4.legend(fontsize=7, ncol=2)
            _ax4.grid(True, alpha=0.3)
            _fig4.tight_layout()
            _show(_fig4)


_run_btn.on_click(_run_eval)

# Clear any VBoxes left over from previous runs of this cell before displaying.
clear_output(wait=True)
display(widgets.VBox([
    _top_row,
    _controls_body,
    _run_btn,
    _plot_out,
]))

In [ ]:
import parse_training_log

_sfx  = '_mlx' if backend == 'mlx' else ''
_odir = eval_cfg.get('odir', './out')
_p    = Path(_odir) / f'metrics{_sfx}.json'
if _p.exists():
    _hist = parse_training_log.TrainingHistory.load(str(_p))
    _hist.plot(title=f'FNO Training ({"MLX" if backend == "mlx" else "PyTorch"})')
    _fig_log = plt.gcf()
    _fig_log.tight_layout()
    _show(_fig_log)
else:
    print(f'No {_p.name} found at {_p}')